In [ ]:
!pip install sentence-transformers

In [ ]:
!pip install rank_bm25

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
# Embedder Model
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
tokenizer = model.tokenizer

In [ ]:
# NEED TO IMPORT CROSS ENCODER AND A NEW MODEL FOR RERANKER
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

In [68]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 55.4 MB/s eta 0:00:00


In [ ]:
!git clone --depth 1 https://github.com/Unstructured-IO/unstructured.git

In [ ]:
from typing import List, Dict, Tuple, Optional

I will reuse old code Only adding HyDE , MMR and multi doc handling + Either Dense Index or Vector DB


In [ ]:
!pip install PyMuPDF          # PDF parsing (import fitz)
!pip install beautifulsoup4   # HTML parsing
!pip install lxml             # faster HTML/XML parser backend for BS4
!pip install markdown         # Markdown → HTML (optional, if you want to strip MD syntax)
!pip install python-docx

In [6]:

from pathlib import Path
import xml.etree.ElementTree as ET
# PDF parsing
import pymupdf
# HTML parsing
from bs4 import BeautifulSoup
# DOCX parsing
from docx import Document
from typing import Callable, Dict, List, Optional, Union

In [ ]:
# Multi Doc loader(pdf,xml,html,txt,md,ms word files all accepted)


In [35]:
!mk dir test_docs_2
!unzip -o test_docs_2.zip

/bin/bash: line 1: mk: command not found
Archive:  test_docs_2.zip
  inflating: test_docs/sample.docx   
  inflating: test_docs/sample.htm    
  inflating: test_docs/sample.html   
  inflating: test_docs/sample.md     
  inflating: test_docs/sample.pdf    
  inflating: test_docs/sample.txt    
  inflating: test_docs/sample.xml    


In [36]:
import os

new_dir_path = "new_test_docs_2"
if not os.path.exists(new_dir_path):
    os.makedirs(new_dir_path)

!unzip -o test_docs_2.zip -d {new_dir_path}

Archive:  test_docs_2.zip
   creating: new_test_docs_2/test_docs/
  inflating: new_test_docs_2/test_docs/sample.docx  
  inflating: new_test_docs_2/test_docs/sample.htm  
  inflating: new_test_docs_2/test_docs/sample.html  
  inflating: new_test_docs_2/test_docs/sample.md  
  inflating: new_test_docs_2/test_docs/sample.pdf  
  inflating: new_test_docs_2/test_docs/sample.txt  
  inflating: new_test_docs_2/test_docs/sample.xml  


In [29]:


class DataLoader:
    def __init__(self):
        self._parsers: Dict[str, Callable[[str], str]] = {
            ".pdf":  self._parse_pdf,
            ".html": self._parse_html,
            ".htm":  self._parse_html,
            ".md":   self._parse_md,
            ".xml":  self._parse_xml,
            ".txt":  self._parse_txt,
            ".docx": self._parse_docx,
        }

    def load(self, path: Union[str, Path]) -> Dict[str, str]:
        file_path = Path(path)
        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {path}")
        if not file_path.is_file():
            raise ValueError(f"Not a file: {path}")

        ext = file_path.suffix.lower()
        parser = self._parsers.get(ext)
        if parser is None:
            raise ValueError(f"Unsupported file extension: {ext}")

        text = parser(str(file_path))
        return {file_path.name: text}

    def load_directory(
        self,
        dir_path: Union[str, Path],
        extensions: Optional[List[str]] = None,
        recursive: bool = False,
        on_error: str = "warn",  # "warn" | "raise" | "ignore"
    ) -> Dict[str, str]:

        dir_path = Path(dir_path)
        if not dir_path.exists():
            raise FileNotFoundError(f"Directory not found: {dir_path}")
        if not dir_path.is_dir():
            raise ValueError(f"Not a directory: {dir_path}")

        if extensions is not None:
            allowed = {self._normalize_ext(e) for e in extensions}
            allowed &= set(self._parsers.keys())
        else:
            allowed = set(self._parsers.keys())

        result: Dict[str, str] = {}
        iterator = dir_path.rglob("*") if recursive else dir_path.iterdir()
        for item in iterator:
            if not item.is_file():
                continue
            if item.name.startswith("."):
                continue
            ext = item.suffix.lower()
            if ext not in allowed:
                continue
            doc_id = item.name
            if doc_id in result:
                print(f"Warning: duplicate doc_id '{doc_id}', overwriting")
            try:
                loaded = self.load(item)
            except Exception as e:
                if on_error == "raise":
                    raise
                if on_error == "warn":
                    print(f"Warning: failed to load {item.name}: {e}")
                continue
            result.update(loaded)
        return result

    def register(self, ext: str, parser_fn: Callable[[str], str]) -> None:
        if not callable(parser_fn):
            raise TypeError(f"parser_fn must be callable, got {type(parser_fn).__name__}")
        self._parsers[self._normalize_ext(ext)] = parser_fn

    @staticmethod
    def _normalize_ext(ext: str) -> str:
        ext = ext.lower().strip()
        return ext if ext.startswith(".") else "." + ext

    @staticmethod
    def _read_text(path: str, encoding: str = "utf-8") -> str:
        try:
            with open(path, "r", encoding=encoding) as f:
                return f.read().strip()
        except UnicodeDecodeError:
            with open(path, "r", encoding=encoding, errors="replace") as f:
                return f.read().strip()

    def _parse_pdf(self, path: str) -> str:
        parts = []
        with pymupdf.open(path) as doc:
            for page in doc:
                parts.append(page.get_text())
        return "\n".join(parts).strip()

    def _parse_html(self, path: str) -> str:
        html_content = self._read_text(path)
        soup = BeautifulSoup(html_content, "lxml")
        for tag in soup(["script", "style"]):
            tag.decompose()
        return soup.get_text(separator="\n", strip=True)

    def _parse_md(self, path: str) -> str:
        return self._read_text(path)
    _parse_txt = _parse_md

    def _parse_xml(self, path: str) -> str:
      try:
          tree = ET.parse(path)
      except ET.ParseError:
          for enc in ("utf-16", "utf-8-sig", "utf-8", "latin-1"):
              try:
                  with open(path, "r", encoding=enc) as f:
                      content = f.read()
                  tree = ET.fromstring(content)
                  break
              except Exception:
                  continue
          else:
              raise

      root = tree.getroot() if hasattr(tree, "getroot") else tree
      parts = []
      for elem in root.iter():
          if elem.text and elem.text.strip():
              parts.append(elem.text.strip())
          if elem.tail and elem.tail.strip():
              parts.append(elem.tail.strip())
      return "\n".join(parts)


    def _parse_docx(self, path: str) -> str:
      try:
          doc = Document(path)
          parts = []
          for para in doc.paragraphs:
              if para.text.strip():
                  parts.append(para.text)
          for table in doc.tables:
              try:
                  for row in table.rows:
                      row_text = " | ".join(
                          cell.text.strip() for cell in row.cells if cell.text.strip()
                      )
                      if row_text:
                          parts.append(row_text)
              except Exception:
                  # skip broken tables
                  continue
          return "\n".join(parts).strip()
      except Exception as e:
          # last resort: try raw XML text extraction
          raise RuntimeError(f"DOCX parse failed: {e}") from e

In [ ]:
# TEST CLASS - 1 DATALOADER

In [ ]:
# loader = DataLoader()
# docs_dir = Path("/content/unstructured/example-docs")   # or "test_docs"

# # 1. Single file
# pdfs = list(docs_dir.glob("*.pdf"))
# if pdfs:
#     result = loader.load(pdfs[0])
#     name, text = next(iter(result.items()))
#     print(f"Loaded: {name}")
#     print(f"Length: {len(text)} chars")
#     print(text[:300], "...\n")

# # 2. Whole directory
# docs = loader.load_directory(
#     docs_dir,
#     extensions=[".pdf", ".docx", ".txt", ".md", ".html", ".xml"],
#     recursive=False,
#     on_error="warn",
# )

# print(f"\nTotal documents loaded: {len(docs)}")
# for name, text in docs.items():
#     print(f"  {name}: {len(text)} chars")

# docs['example-10k-230p.html']

In [ ]:
# docs['example-10k-230p.html']

In [ ]:

DIR_PATH = "/content/ALL_FORMAT_DIR"


loader = DataLoader()
docs = loader.load_directory(DIR_PATH)

print(f"\nLoaded {len(docs)} documents:\n")
for doc_id, text in docs.items():
    print(f"📄 {doc_id}  |  {len(text)} chars")
    print(f"   Preview: {text[:120]}...\n")


Loaded 6 documents:

📄 photosynthesis.txt  |  1810 chars
   Preview: Photosynthesis is the process by which plants convert light energy into chemical energy. It occurs primarily in the chlo...

📄 photosynthesis.pdf  |  1833 chars
   Preview: Photosynthesis is the process by which plants convert light energy into chemical energy. It occurs 
primarily in the chl...

📄 immune_system.xml  |  1626 chars
   Preview: Immune System Overview
Biology Department
2024-01-15
Introduction
The immune system is the body's primary defense networ...

📄 http_protocol.md  |  1590 chars
   Preview: # HTTP Protocol

## Overview
HTTP, or Hypertext Transfer Protocol, is the foundational protocol used for communication o...

📄 black_holes.html  |  1024 chars
   Preview: Black Holes
Black Holes
A black hole is a region of spacetime where gravity is so strong that nothing, not even light, c...

📄 photosynthesis.docx  |  1806 chars
   Preview: Photosynthesis is the process by which plants convert light energy in

In [ ]:
docs['black_holes.html']

"Black Holes\nBlack Holes\nA black hole is a region of spacetime where gravity is so strong that nothing, not even light, can escape once it crosses the event horizon.\nBlack holes form when massive stars collapse under their own gravity at the end of their life cycle, typically after a supernova explosion.\nTypes of Black Holes\nStellar black holes have masses several times greater than the Sun. Supermassive black holes, containing millions or even billions of solar masses, are believed to exist at the center of most large galaxies, including our own Milky Way.\nThis paragraph should be hidden in a browser but the parser might still extract it — that's okay.\nHawking Radiation\nHawking radiation is a theoretical process through which black holes slowly lose mass and energy over immense timescales, eventually evaporating entirely. This phenomenon arises from quantum effects occurring near the event horizon.\nNested paragraph: The event horizon marks the boundary beyond which escape fro

NOW REUSE OLD CODE


In [ ]:
# NO USE OF INPUT_PROCESSING AND GET_DICT FUNC AS I ALREADY HAVE A DICT[DOC_ID] = RAW TEXT(UNPROCESSES)
# DIRECTLY USE THE CHUNKER ON EVERY DOCS

# def input_processing(corpus_str=None, para_id_list=None, corpus_dict=None, isDict=False, isParaSplit=False):
#     if corpus_str is None and corpus_dict is None:
#         raise Exception("No Corpus Given")
#     if isDict:
#         if corpus_dict is None:
#             raise Exception("isDict=True but no corpus_dict given")
#         return list(corpus_dict.keys()), list(corpus_dict.values())
#     if isParaSplit:
#         pre_pro_docs = corpus_str.split("\n\n")
#     else:
#         pre_pro_docs = corpus_str.split('\n')
#     output_corpus = [doc.strip() for doc in pre_pro_docs if doc.strip()]
#     if para_id_list is not None:
#         if len(para_id_list) != len(output_corpus):
#             raise ValueError(
#                 f"Length mismatch AFTER filtering: {len(para_id_list)} ids vs "
#                 f"{len(output_corpus)} non-empty docs — check your split logic "
#                 f"or para_ids ordering"
#             )

#     print("Length of output corpus list ", len(output_corpus))
#     return para_id_list, output_corpus

In [63]:
class Chunker:
    def __init__(self, tokenizer, max_length=254, overlap=32):
        if not tokenizer:
            raise Exception("No Tokenizer Given")
        if overlap >= max_length:
            raise ValueError(f"Overlap ({overlap}) cannot be greater than max_length ({max_length})")

        self.tokenizer = tokenizer
        self.max_length = max_length
        self.overlap = overlap

    def _token_len(self, text):
        if not text or not text.strip():
            return 0
        return len(self.tokenizer.encode(text, add_special_tokens=False))

    def _decode_token(self, token_ids):
        return self.tokenizer.decode(token_ids)

    def _split_text(self, text, sep="\n"):
        return [s for s in text.split(sep) if s.strip()]
    # Handle case of the text _token_overlap_text takes a finished chunk and returns only the last N tokens of that text, cut from the original string (so casing and pacing stay the same).
    # We need this because encode→decode on an uncased model would lowercase the overlap. Using character offsets from the original text avoids that.

    def _token_overlap_text(self, text: str) -> str:
      if not text or self.overlap <= 0:
          return ""

      encoded = self.tokenizer(
          text,
          add_special_tokens=False,
          return_offsets_mapping=True,
      )
      offsets = encoded["offset_mapping"]
      if not offsets:
          return ""

      if len(offsets) <= self.overlap:
          return text

      start_char = offsets[-self.overlap][0]
      return text[start_char:]
# Emit current buffer as a chunk. If overlap is enabled, seed temp with the last overlap tokens of that chunk so the next chunk shares a boundary with the previous one (avoids starting empty when the previous chunk was already near max_length).

    def _merge_chunks(self, chunks, overlap=False):
      if not chunks:
          return []

      merged = []
      temp = []
      temp_len = 0

      def emit_and_seed_overlap():
        nonlocal temp, temp_len
        if not temp:
            return

        text = " ".join(temp)
        merged.append(text)

        if overlap and self.overlap > 0:
            overlap_text = self._token_overlap_text(text).strip()
            if overlap_text:
                temp = [overlap_text]
                temp_len = self._token_len(overlap_text)
            else:
                temp = []
                temp_len = 0
        else:
            temp = []
            temp_len = 0

      for chunk in chunks:
          chunk_len = self._token_len(chunk)
          if chunk_len == 0:
              continue

          # single piece already too big → fixed-size split
          if chunk_len > self.max_length:
              if temp:
                  emit_and_seed_overlap()
              # use overlapping fixed-size when overlap=True
              if overlap:
                  merged.extend(self.fixed_size_overlap(chunk))
              else:
                  merged.extend(self.fixed_size_no_overlap(chunk))
              temp = []
              temp_len = 0
              continue

          sep_len = 1 if temp else 0

          if temp_len + sep_len + chunk_len <= self.max_length:
              temp.append(chunk)
              temp_len += sep_len + chunk_len
          else:
              # current buffer is full → emit it (and seed overlap)
              emit_and_seed_overlap()

              # try to add the new chunk on top of the overlap seed
              sep_len = 1 if temp else 0
              if temp_len + sep_len + chunk_len <= self.max_length:
                  temp.append(chunk)
                  temp_len += sep_len + chunk_len
              else:
                  # still does not fit (overlap + chunk too big) → start fresh
                  temp = [chunk]
                  temp_len = chunk_len

      if temp:
          merged.append(" ".join(temp))

      return merged
    def _recursive_split(self, text, separators=("\n\n\n", "\n\n", "\n", "\t", ". ", " ")):
        if self._token_len(text) <= self.max_length:
            return [text]

        if not separators:
            return self.fixed_size_no_overlap(text)

        sep = separators[0]
        splits = self._split_text(text, sep)
        result = []

        for split in splits:
            if self._token_len(split) <= self.max_length:
                result.append(split)
            else:
                result.extend(self._recursive_split(split, separators[1:]))

        return result

    def recursive_chunk(self, text):
        splits = self._recursive_split(text)
        return self._merge_chunks(splits, overlap=False)

    def recursive_chunk_overlap(self, text):
        splits = self._recursive_split(text)
        return self._merge_chunks(splits, overlap=True)

    def fixed_size_no_overlap(self, text):
        tokens = self.tokenizer.encode(text, add_special_tokens=False)
        if not tokens:
            return []
        chunks = []
        for start in range(0, len(tokens), self.max_length):
            window = tokens[start:start + self.max_length]
            chunks.append(self.tokenizer.decode(window))
        return chunks

    def fixed_size_overlap(self, text):
        tokens = self.tokenizer.encode(text, add_special_tokens=False)
        if not tokens:
            return []
        chunks = []
        step = self.max_length - self.overlap
        for start in range(0, len(tokens), step):
            window = tokens[start:start + self.max_length]
            chunks.append(self.tokenizer.decode(window))
            if start + self.max_length >= len(tokens):
                break
        return chunks

    def chunk(self, text, method="fixed_size_no_overlap"):
        if method == "fixed_size_no_overlap":
            return self.fixed_size_no_overlap(text)
        elif method == "fixed_size_overlap":
            return self.fixed_size_overlap(text)
        elif method == "recursive_no_overlap":
            return self.recursive_chunk(text)
        elif method == "recursive_overlap":
            return self.recursive_chunk_overlap(text)
        else:
            raise Exception(f"Wrong Method: {method}")




In [55]:
def build_full_dict_per_doc_id(chunker, text, doc_id, method='fixed_size_no_overlap'):
    if not chunker:
        raise Exception("No chunker was given")
    if not doc_id:
        raise Exception("No doc_id was given")

    chunks_per_doc = chunker.chunk(text, method)
    result = list()

    for i, chunk_text in enumerate(chunks_per_doc):
        result.append({
            "chunk_text": chunk_text,
            "chunk_id": f"{doc_id}_{method}_{i}",
            "doc_id": doc_id,
            "chunk_index": i,
            "chunking_method": method,
            "token_count": chunker._token_len(chunk_text),  # STILL ACCESSING THE SEMI PRIVATE METHOD IN CHUNKER CLASS IDTS ANY CODE WILL BREAK BECAUSE OF THIS
        })

    return result

In [56]:
def process_all_docs_metadata(chunker, corpus_dict, method='fixed_size_no_overlap'):
    if not chunker:
        raise Exception("No chunker given")
    if not corpus_dict:
        raise Exception("No corpus_dict given")

    all_records = []
    for doc_id, text in corpus_dict.items():
        all_records.extend(build_full_dict_per_doc_id(chunker, text, doc_id, method))

    return all_records

In [40]:
loader = DataLoader()

dir_path = Path('/content/new_test_docs_2/test_docs')

In [ ]:
docs = loader.load_directory(dir_path)
docs

In [ ]:
# # Test CHUNKER CLASS and build_full_dict_per_doc_id FUNC  and process_all_docs_metadata FUNC

# chunker = Chunker(tokenizer=tokenizer, max_length=254, overlap=32)

# # 2. Chunk all docs
# records = process_all_docs_metadata(
#     chunker=chunker,
#     corpus_dict=docs,
#     method='recursive_overlap'
# )

# # 3. Inspect
# print(f"Total chunks: {len(records)}\n")
# for r in records:
#     print(f"doc_id: {r['doc_id']}")
#     print(f"chunk_id: {r['chunk_id']}")
#     print(f"tokens: {r['token_count']}")
#     print(f"text: {r['chunk_text'][:120]}...")
#     print("-" * 60)


# c0 = records[0]["chunk_text"]      # chunk 0
# c1 = records[1]["chunk_text"]    # chunk 1

# print("END of chunk 0:")
# print(c0[-200:])
# print("\nSTART of chunk 1:")
# print(c1[:200:])


# end0 = c0[-80:].strip()
# start1 = c1[:80].strip()

# # True overlap only if some non-trivial suffix of c0 appears at the start of c1
# print(end0 in c1 or any(
#     c0.endswith(c1[:k]) for k in range(40, min(120, len(c1)), 10)
# ))


In [65]:
# records

OLD CODE FOR BUILDING INDICES + Adding HyDE and Adding FAISS ALL THE 3 are interchangeable modular func


In [70]:
import faiss
from rank_bm25 import BM25Okapi

# shape (num_chunks, dim) — aligned by index with records
# I include the creation of faiss index within this func using a bool flag param
def build_dense_index(model=None, records=None, return_faiss_index=False):
  if model is None or records is None: raise Exception("No Model or Records Given")
  texts = [r['chunk_text'] for r in records]
  embeds = model.encode(texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)

  if return_faiss_index:
    dimension = embeds.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeds)
    return index
  else:
    return embeds # DIMS(NO OF CHUNKS , DIMS OF MODEL)


# This func is building a somewhat inverted and forward index combine index for all the tokens in corpus and bm25 naturally leans more towards scoring based on cumulative
# addition of scores(query_term,corpus)
# No vectorization is needed

def build_sparse_index(records=None):
  if records is None: raise Exception("No Records Given")
  texts = [r['chunk_text'] for r in records]
  tokenized = [text.lower().split() for text in texts]
  return BM25Okapi(tokenized)


In [113]:
def hyde_query(query: str, llm_client, model_name="openai/gpt-oss-20b") -> str:
    resp = llm_client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content":
            f"Write a short factual passage that answers:\n{query}"}],
        max_tokens=150, temperature=0.3,
    )
    return resp.choices[0].message.content.strip()

In [74]:
# The dense_search function can now accept either raw embeddings or a FAISS index.
# HyDE will be handled by the higher-level `retrieve` function as a query transformation.
def dense_search(
    query: str,
    model: SentenceTransformer,
    embeddings: Optional[np.ndarray] = None, # For brute-force search if FAISS not used
    faiss_index: Optional[faiss.IndexFlatL2] = None, # For FAISS search
    top_n: int = 10
) -> List[Tuple[int, float]]: # Returns list of (index, score)

    if embeddings is None and faiss_index is None:
        raise ValueError("Either 'embeddings' or 'faiss_index' must be provided for dense_search.")
    if embeddings is not None and faiss_index is not None:
        raise ValueError("Cannot provide both 'embeddings' and 'faiss_index' to dense_search. Choose one.")
    query_emb = model.encode([query], normalize_embeddings=True)
    query_emb = query_emb.astype('float32') # FAISS typically works with float32
    if faiss_index:
        # Perform search using FAISS index
        D, I = faiss_index.search(query_emb, top_n)
        results = []
        for dist, idx in zip(D[0], I[0]):
            if idx != -1: # Ensure valid index
                # Convert FAISS L2 distance to similarity (smaller distance = higher similarity)
                similarity_score = 1.0 / (1.0 + dist)
                results.append((int(idx), float(similarity_score)))
        results.sort(key=lambda x: x[1], reverse=True) # Sort by similarity in descending order
        return results
    else: # Use brute-force with provided embeddings (embeddings must be present)
        if embeddings.dtype != np.float32:
            embeddings = embeddings.astype(np.float32)
        scores = (query_emb @ embeddings.T).flatten()
        top_indices = np.argsort(-scores)[:top_n] # Sort descending for similarity
        return [(int(idx), float(scores[idx])) for idx in top_indices]

def sparse_search(query = None ,bm25_idx = None , top_n=10):
    if query is None or bm25_idx is None:
        raise Exception("query or bm25_idx is None")
    query_tokens = query.lower().split()
    scores = bm25_idx.get_scores(query_tokens)
    top_indices = np.argsort(-scores)[:top_n]
    return [(idx, scores[idx]) for idx in top_indices]


In [95]:
# Addind top_m param to restrict the output of the rrf func to only default top 10 fused results
def rrf(ranked_lists, k=60, top_m=7):
    rrf_scores = {}
    for ranked_list in ranked_lists:
        for rank, (idx, _) in enumerate(ranked_list, start=1):
            rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k + rank)
    fused = sorted(rrf_scores.items(), key=lambda x: -x[1])
    if top_m is not None:
        return fused[:top_m] # [(idx, rrf_score), ...]
    return fused

In [76]:
# This rerank func works only on the top_n fused rank list and re embeds , NSP STYLE LOSS FUNC TO RE RANK THEN : HELPS IN PRECISION IMPROVEMENT
# candidates: list of record dicts (from your metadata), already RRF-fused"
def rerank(query, candidates, top_k=5):
    pairs = [(query, c["chunk_text"]) for c in candidates]
    scores = cross_encoder.predict(pairs)   # one relevance score per pair
    reranked = sorted(zip(candidates, scores), key=lambda x: -x[1])
    return [c for c, s in reranked[:top_k]]

In [108]:
from sklearn.metrics.pairwise import cosine_similarity

def mmr_rerank(
    query: str,
    candidates: List[Dict], # List of record dicts
    model: SentenceTransformer,
    top_k: int = 5,
    diversity: float = 0.5 # 0 = no diversity, 1 = max diversity
) -> List[Dict]:
    if not candidates:
        return []

    # Encode query and candidate texts
    query_embedding = model.encode([query], normalize_embeddings=True)
    candidate_embeddings = model.encode([c["chunk_text"] for c in candidates], normalize_embeddings=True)

    # Calculate similarity between query and candidates
    query_candidate_similarities = cosine_similarity(query_embedding, candidate_embeddings)[0]

    selected_indices = []
    unselected_indices = list(range(len(candidates)))

    # Select the first document (most similar to query)
    first_doc_idx = np.argmax(query_candidate_similarities)
    selected_indices.append(first_doc_idx)
    unselected_indices.remove(first_doc_idx)

    for _ in range(min(top_k, len(candidates)) - 1): # -1 because we already selected one
        if not unselected_indices:
            break

        max_mmr_score = -np.inf
        best_idx_to_add = -1

        selected_embeddings = candidate_embeddings[selected_indices]

        for idx_to_add in unselected_indices:
            candidate_embedding = candidate_embeddings[[idx_to_add]]
            # Calculate diversity score: max similarity to already selected documents
            if len(selected_embeddings) > 0:
              diverse_score = np.max(cosine_similarity(candidate_embedding, selected_embeddings)[0])
            else:
              diverse_score = 0 # No diversity penalty if no documents selected yet

            # MMR formula
            mmr_score = (diversity * query_candidate_similarities[idx_to_add]) - ((1 - diversity) * diverse_score)

            if mmr_score > max_mmr_score:
                max_mmr_score = mmr_score
                best_idx_to_add = idx_to_add

        if best_idx_to_add != -1:
            selected_indices.append(best_idx_to_add)
            unselected_indices.remove(best_idx_to_add)
        else:
            break # No valid candidate found

    # Return the selected candidate documents
    return [candidates[i] for i in selected_indices]

In [ ]:
records

In [83]:
dense_idx_np = build_dense_index(model=model, records=records, return_faiss_index=False)
dense_idx_faiss = build_dense_index(model=model, records=records, return_faiss_index=True)
sparse_idx = build_sparse_index(records=records)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [88]:
# query = 'What is Machine Learning and transfer learning'

In [ ]:
# print(f"Query: {query}\n")

# # Perform Sparse Search
# sparse_results = sparse_search(
#     query=query,
#     bm25_idx=sparse_idx,
#     top_n=10
# )
# print("--- Sparse Search Results ---")
# for i, (idx, score) in enumerate(sparse_results):
#     chunk = records[idx]
#     print(f"  {i+1}. Doc ID: {chunk['doc_id']}, Chunk Index: {chunk['chunk_index']}, Score: {score:.4f}")
#     print(f"     Text: {chunk['chunk_text'][:100]}...\n")

# # Perform Dense Search (with FAISS)
# dense_results_faiss = dense_search(
#     query=query,
#     model=model,
#     faiss_index=dense_idx_faiss,
#     top_n=10
# )
# print("--- Dense Search Results (with FAISS) ---")
# for i, (idx, score) in enumerate(dense_results_faiss):
#     chunk = records[idx]
#     print(f"  {i+1}. Doc ID: {chunk['doc_id']}, Chunk Index: {chunk['chunk_index']}, Score: {score:.4f}")
#     print(f"     Text: {chunk['chunk_text'][:100]}...\n")

# # Perform Dense Search (without FAISS, using numpy embeddings)
# dense_results_np = dense_search(
#     query=query,
#     model=model,
#     embeddings=dense_idx_np,
#     top_n=10
# )
# print("--- Dense Search Results (without FAISS, using NumPy) ---")
# for i, (idx, score) in enumerate(dense_results_np):
#     chunk = records[idx]
#     print(f"  {i+1}. Doc ID: {chunk['doc_id']}, Chunk Index: {chunk['chunk_index']}, Score: {score:.4f}")
#     print(f"     Text: {chunk['chunk_text'][:100]}...\n")

# # Apply Reciprocal Rank Fusion
# # The reciprocal_rank_fusion function expects a list of ranked lists
# ranks_to_fuse = [sparse_results, dense_results_faiss, dense_results_np]
# fused_results = rrf(ranks_to_fuse)

# print("--- Reciprocal Rank Fusion Results ---")
# print(f"Top {len(fused_results)} Fused Results:")
# for i, (idx, rrf_score) in enumerate(fused_results):
#     chunk = records[idx]
#     print(f"  {i+1}. Doc ID: {chunk['doc_id']}, Chunk Index: {chunk['chunk_index']}, RRF Score: {rrf_score:.4f}")
#     print(f"     Text: {chunk['chunk_text'][:150]}...\n")

In [117]:
from typing import Dict, Any, Optional, List

def retrieve(
    query: str,
    strategy_data: Dict, # should contain records, dense_embeddings or faiss_index, bm25_index, and optionally llm
    model: SentenceTransformer, # Explicitly pass the model for MMR and dense_search
    top_n: int = 20, # Initial candidates before RRF/reranker
    final_k: int = 5, # Final number of documents after reranking
    use_reranker: bool = True,
    use_hyde: bool = False,
    use_faiss: bool = False, # New parameter to control FAISS usage
    rrf_top_m: Optional[int] = None, # New parameter for rrf top_m
    use_mmr: bool = False, # New parameter for MMR
    mmr_diversity: float = 0.5 # New parameter for MMR diversity
) -> List[Dict]:

    processed_query = query
    if use_hyde:
        if "llm" not in strategy_data:
            raise ValueError("LLM not provided in strategy_data for HyDE.")
        processed_query = hyde_query(query, strategy_data["llm"])

    # Perform Dense Search
    if use_faiss:
        if "faiss_index" not in strategy_data:
            raise ValueError("use_faiss=True but no faiss_index provided.")
        dense_results = dense_search(processed_query, model,
                                    faiss_index=strategy_data["faiss_index"], top_n=top_n)
    else:
        if "dense_embeddings" not in strategy_data:
            raise ValueError("use_faiss=False but no dense_embeddings provided.")
        dense_results = dense_search(processed_query, model,
                                    embeddings=strategy_data["dense_embeddings"], top_n=top_n)
    # Perform Sparse Search
    sparse_results = sparse_search(processed_query, strategy_data["bm25_index"], top_n=top_n)

    # Apply Reciprocal Rank Fusion
    # If rrf_top_m is not explicitly set, use top_n as the shortlist size before reranking
    fused = rrf([dense_results, sparse_results], top_m=rrf_top_m if rrf_top_m is not None else top_n)

    # Get candidates from records based on fused results
    candidates = [strategy_data["records"][idx] for idx, _ in fused]

    if use_reranker:
        # If MMR was used, rerank the MMR-selected candidates.
        # If MMR was NOT used, rerank the RRF-selected candidates.
        return rerank(query, candidates, top_k=final_k)


    # Apply MMR if requested, before final reranking or truncation
    if use_mmr:
        candidates = mmr_rerank(
            query=query,
            candidates=candidates,
            model=model,
            top_k=final_k, # MMR reduces to final_k or fewer
            diversity=mmr_diversity
        )


    else:
        # If reranker is not used, and MMR was used, return MMR-selected candidates.
        # If neither MMR nor reranker was used, return RRF-selected candidates, truncated to final_k.
        return candidates[:final_k]

In [97]:
def build_prompt(raw_query,results):
  context = "\n\n".join(r['chunk_text'] for r in results)

  prompt = f"""Answer the question using ONLY the context below. If the context doesn't contain the answer, say so — do not use outside knowledge.
Context:
{context}
Question: {raw_query}
Answer:"""

  return prompt

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="api_key"
)

def generate_answer(prompt, client, model_name="openai/gpt-oss-20b"):
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [110]:
def RAG_Complete(
    raw_query: str,
    bm25_index: BM25Okapi,
    records: list,
    model: SentenceTransformer,
    client: OpenAI, # Client for generate_answer
    dense_embeddings: Optional[np.ndarray] = None, # Make optional
    faiss_index: Optional[faiss.IndexFlatL2] = None, # New parameter
    llm: Optional[Any] = None, # New parameter for HyDE (e.g., PlaceholderLLM)
    k: int = 5, # Corresponds to final_k in retrieve (number of final documents)
    use_reranker: bool = True,
    use_hyde: bool = False, # New parameter
    use_faiss: bool = False, # New parameter
    top_n_retrieval: int = 20, # Corresponds to top_n in retrieve (number of initial candidates)
    rrf_top_m: Optional[int] = None, # New parameter for rrf top_m
    use_mmr: bool = False, # New parameter for MMR
    mmr_diversity: float = 0.5 # New parameter for MMR diversity
) -> str:

    # Assemble strategy_data for the retrieve function
    strategy_data = {
        "records": records,
        "bm25_index": bm25_index,
    }

    if dense_embeddings is not None:
        strategy_data["dense_embeddings"] = dense_embeddings
    if faiss_index is not None:
        strategy_data["faiss_index"] = faiss_index
    if llm is not None:
        strategy_data["llm"] = llm

    # Retrieve documents
    retrieved_results = retrieve(
        raw_query,
        strategy_data,
        model, # Pass the model explicitly
        top_n=top_n_retrieval,
        final_k=k,
        use_reranker=use_reranker,
        use_hyde=use_hyde,
        use_faiss=use_faiss,
        rrf_top_m=rrf_top_m,
        use_mmr=use_mmr,
        mmr_diversity=mmr_diversity
    )

    # Build the prompt
    prompt = build_prompt(raw_query, retrieved_results)

    # Generate the answer
    final_answer = generate_answer(prompt, client)

    return final_answer

In [118]:
query = 'What is machine learning and carbon capturing and who was first emperor of roman empire'

# Call the RAG_Complete function using the pre-built indices and models
final_rag_answer_full = RAG_Complete(
    raw_query=query,
    dense_embeddings=dense_idx_np, # Using numpy embeddings (assuming use_faiss=False)
    bm25_index=sparse_idx,
    records=records,
    model=model,
    client=client,
    llm = client,
    k=4,                  # Request 4 final documents
    use_reranker=True,    # Use reranker (default)
    use_hyde=False,       # Do not use HyDE (default, as no LLM was passed)
    use_faiss=False,
    use_mmr = True # Do not use FAISS (default)
)

print(f"\nQuery: {query}")
print(f"Answer: {final_rag_answer_full}")


Query: What is machine learning and carbon capturing and who was first emperor of roman empire
Answer: **Machine learning**  
Machine learning enables systems to learn patterns from data without explicit programming. The main paradigms are supervised learning (using labeled examples to map inputs to outputs), unsupervised learning (discovering structure in unlabeled data through clustering and dimensionality reduction), and reinforcement learning (training agents to make sequences of decisions by maximizing cumulative reward).

**Carbon capture**  
Carbon capture technologies include direct air capture and point‑source carbon capture, which are being piloted at industrial scale. They aim to remove CO₂ from the atmosphere or from emissions streams and store it in deep geological formations or use it to produce synthetic fuels and building materials. Life‑cycle assessments, monitoring, reporting, and verification systems are essential to ensure real and measurable climate benefit.

**Fi